In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split




read_file = pd.read_excel('./fmri_map_id.xlsx')
subjects = list(read_file['BIDS_ID'])





df_ma = pd.read_excel('./MATERIAL.xlsx')
id_2_dur=dict(zip(df_ma['TrialID'],df_ma['soundDur']))


trail_list = [ str(df_ma['word1'][i]) + ' ' + str(df_ma['word2'][i]) + ' '  + str(df_ma['word3'][i]) + ' ' + str(df_ma['word4'][i]) + ' '   
              + str(df_ma['word5'][i]) + ' '   + str(df_ma['word6'][i]) + ' '   + str(df_ma['word7'][i]) + ' '   + str(df_ma['word8'][i])
               for i in range(len(df_ma))]
trail_list = [sentence.replace("nan", "").strip() for sentence in trail_list]


def remove_duplicates(sentence):
    words = sentence.split() 
    for i in range(7):
        if words[-1] ==words[-2]:
            words.pop(-1)
    return ' '.join(words)  


new_list = [remove_duplicates(sentence) for sentence in trail_list]

id_2_trail_type = dict(zip(list(df_ma['TrialID']),new_list))




file_path = './output.txt'

with open(file_path, 'r', encoding='utf-8') as file:
    lines = file.readlines()


item_listxt = [line.strip() for line in lines]


new_list_co = []
new_list_unc = []
for co in new_list:
    if co in item_listxt:
        new_list_co.append(co)
    else:
        new_list_unc.append(co)

new_list = new_list_co + new_list_unc



count_dict = {}
count_new_list = []


for sentence in new_list:
    if sentence in count_dict:
        
        count_dict[sentence] += 1
    else:
        
        count_dict[sentence] = 0
    
    
    count_new_list.append(f"{sentence} {count_dict[sentence]}")


class AutoEncoder(nn.Module):
    def __init__(self):
        super(AutoEncoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(271633, 768),
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(768, 271633),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for sub_id,  subject in enumerate(subjects):

    test_indices = [sub_id]
    print(subjects[sub_id])
    train_indices = list(range(len(subjects)))
    train_indices.remove(sub_id)


    #train_indices, test_indices = train_test_split(range(len(all_sub_vec)), test_size=0.2, random_state=42)


    autoencoder = AutoEncoder().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)


    num_epochs = 20
    batch_size = 64  


    autoencoder_list = []
    test_loss_list = []
    encoded_np_list = []
    save_mark = 0
    patience =3
    no_improve_count = 0
 
    for epoch in range(num_epochs):
        print('started')
        for idx in train_indices:
          
            #print(subjects[idx])
            vector = torch.Tensor(np.load(f'./neural_data_all_brain_no_reduced_mni/sub-{subjects[idx]}_neural_vector.npy')).to(device) 

            # batches
            dataset = torch.utils.data.TensorDataset(vector)
            dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

            for batch in dataloader:
                inputs = batch[0].to(device) 
                optimizer.zero_grad()

                
                encoded, decoded = autoencoder(inputs)

                
                loss = criterion(decoded, inputs)
                loss.backward() 
                optimizer.step() 

        autoencoder_list.append(autoencoder.state_dict())
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

      
        with torch.no_grad():
            total_loss = 0

            vector = torch.Tensor(np.load(f'./neural_data_all_brain_no_reduced_mni/sub-{subject}_neural_vector.npy')).to(device)
       
            encoded, decoded = autoencoder(vector)
            loss = criterion(decoded, vector)
            total_loss += loss.item()
            encoded_np = encoded.detach().cpu().numpy()
            encoded_np_list.append(encoded_np)
            test_loss_list.append(total_loss)
            print(f"Test Loss: {total_loss / len(test_indices):.4f}")
            
        if ( epoch > 0 ) and (total_loss > min(test_loss_list)):
            no_improve_count += 1
            if no_improve_count >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                min_loss_index = test_loss_list.index(min(test_loss_list))
                
                np.save(f'./sub-{subject}/sub-{subject}_neural_vector.npy', 
                        encoded_np_list[min_loss_index])
                torch.save(autoencoder_list[min_loss_index],
                        f'./sub-{subject}/autoencoder.pth')
                save_mark = 1
                break



    if save_mark == 0:
        # save the last result
        min_loss_index = test_loss_list.index(min(test_loss_list))
        np.save(f'./sub-{subject}_neural_vector.npy', 
                        encoded_np_list[min_loss_index])
        torch.save(autoencoder_list[min_loss_index], 
                        f'./sub-{subject}/autoencoder.pth')


